In [2]:
# 📚 导入必要的库

import sys
import os
import time
import gc
from pathlib import Path
from typing import Optional
import numpy as np
import xarray as xr
import intake
# 导入自定义工具
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools.utils import dataarray_healpix_to_equatorial_latlon

print("✅ 所有库已导入")

✅ 所有库已导入


In [4]:
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
cat

data_nextgems-h2020_eu:
  args:
    path: https://data.nextgems-h2020.eu/catalog.yaml
  description: ''
  driver: intake.catalog.local.YAMLFileCatalog
  metadata: {}


In [5]:
cat['ICON']

ICON:
  args:
    path: https://data.nextgems-h2020.eu/ICON/main.yaml
  description: ICON model output
  driver: intake.catalog.local.YAMLFileCatalog
  metadata:
    catalog_dir: https://data.nextgems-h2020.eu


In [ ]:
# 🔧 工具函数

def dataarray_to_equatorial_latlon_grid(
    dataarray: xr.DataArray, grid_type: str, grid_dict: Optional[dict]
) -> xr.DataArray:
    """转换数据到赤道经纬度网格"""
    if grid_type == "latlon":
        return dataarray
    elif grid_type == "healpix":
        if grid_dict is None:
            raise ValueError("No grid_dict provided for healpix conversion.")
        return dataarray_healpix_to_equatorial_latlon(dataarray, **grid_dict)
    else:
        raise ValueError("Grid type not found.")

print("✅ 工具函数已加载")

In [ ]:
# 🎯 核心处理函数

def process_var_data(var_name, experiment_name, dataset_key, save_dir, grid_dict, 
                     target_lat, target_lon, has_level=True, level_slice=(None, None), 
                     max_levels=None, cat=None):
    """
    处理变量数据 - 优化版本
    
    使用本地线程计算，避免Dask分布式的内存限制问题
    
    参数:
        max_levels: 最多处理多少层（用于测试），None表示处理所有层
    """
    # 加载catalog
    if cat is None:
        cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
    
    print("="*70)
    print(f"🔄 {var_name.upper()} - {experiment_name}")
    print("="*70)
    
    # 创建保存目录
    exp_save_dir = os.path.join(save_dir, f"{var_name}_3d", experiment_name.lower()) if has_level else os.path.join(save_dir, f"{var_name}_2d", experiment_name.lower())
    os.makedirs(exp_save_dir, exist_ok=True)
    
    if not has_level:
        print("⚠️  2D数据处理功能待实现")
        return
    
    # 3D数据处理
    print("📖 读取数据...")
    var_temp = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(time=slice("1980", "1993"))
    
    # 查找高度维度
    level_dims = ['level_full', 'level_half', 'level', 'plev', 'pressure', 'height', 'lev']
    level_dim = next((dim for dim in level_dims if dim in var_temp.dims), None)
    
    if not level_dim:
        raise ValueError(f"无法找到高度维度！维度: {list(var_temp.dims)}")
    
    print(f"✅ 高度维度: {level_dim}")
    
    var_full = var_temp.sel({level_dim: slice(*level_slice)})
    # 优化分块，减少任务图大小
    var_full = var_full.chunk({'time': 10, level_dim: 1, 'cell': -1})
    
    levels = var_full[level_dim].values
    
    # 如果指定了max_levels，只处理前max_levels层
    if max_levels is not None:
        levels = levels[:max_levels]
    
    n_levels = len(levels)
    
    print(f"✅ 总层数: {n_levels}, 时间步: {len(var_full.time)}")
    
    # 检查需要处理的层
    levels_to_process = [
        lev for lev in levels 
        if not os.path.exists(os.path.join(exp_save_dir, f"{var_name}_lev_{int(lev):03d}.nc"))
    ]
    
    if not levels_to_process:
        print("✅ 所有层已处理完成")
        return
    
    print(f"📊 需处理: {len(levels_to_process)}/{n_levels} 层\n")
    
    total_start = time.time()
    
    # 逐层处理
    for idx, level in enumerate(levels_to_process, 1):
        t0 = time.time()
        save_path = os.path.join(exp_save_dir, f"{var_name}_lev_{int(level):03d}.nc")
        
        print(f"[{idx}/{len(levels_to_process)}] Lev {int(level):3d}", end=' → ', flush=True)
        
        try:
            # 选择单层并使用小分块，立即持久化
            var_layer = var_full.sel({level_dim: level}).chunk({'time': 10, 'cell': -1})
            
            # 转换到经纬度网格（使用分布式计算）
            var_lonlat = dataarray_to_equatorial_latlon_grid(var_layer, 'healpix', grid_dict)
            var_lonlat = var_lonlat.chunk({'time': 10, 'lat': 20, 'lon': 20})
            
            # 分步计算：先持久化到内存，减少任务图复杂度
            var_lonlat = var_lonlat.persist()
            
            # 等待计算完成
            import dask
            dask.distributed.wait(var_lonlat)
            
            # 插值到2度（本地计算小数据）
            var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear').compute()
            
            # 保存
            ds = var_2deg.to_dataset(name=var_name)
            ds.to_netcdf(save_path, encoding={var_name: {'zlib': True, 'complevel': 4}})
            
            # 清理内存
            del var_layer, var_lonlat, var_2deg, ds
            gc.collect()
            
            # 计算时间
            elapsed = time.time() - t0
            total_elapsed = time.time() - total_start
            avg = total_elapsed / idx
            remaining = avg * (len(levels_to_process) - idx)
            
            print(f"✅ {elapsed:.1f}s (剩余: {remaining/60:.1f}min)")
            
        except Exception as e:
            print(f"❌ {str(e)[:50]}")
            gc.collect()
    
    total_time = time.time() - total_start
    print(f"\n✅ 完成! 总耗时: {total_time/60:.1f}分钟")
    print("="*70 + "\n")

print("✅ 核心处理函数已加载")

In [ ]:
# 🧪 测试函数（可选）

def test_single_layer():
    """测试处理前3个压力层"""
    cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
    
    # 配置参数
    var_name = 'rho'
    experiment = 'cntl'
    dataset_key = 'AMIP_CNTL'
    save_dir = '/work/mh1498/m301257/converted_data/processed/'
    
    # 网格和目标分辨率
    grid_dict = {'nside': 256, 'nest': True, 'minmax_lat': 20}
    target_lat = xr.DataArray(np.arange(-20, 21, 2), dims='lat')
    target_lon = xr.DataArray(np.arange(0, 360, 2), dims='lon')
    
    print("🧪 测试模式: 只处理前3层")
    process_var_data(var_name, experiment, dataset_key, save_dir, grid_dict, 
                     target_lat, target_lon, has_level=True, 
                     level_slice=(None, None), max_levels=3,  # 只处理前3层
                     cat=cat)
    print("✅ 测试完成！")

print("✅ 测试函数已加载")

In [ ]:
# 🚀 主执行函数

def main():
    """完整数据处理流程"""
    cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
    
    # 配置
    var_name = 'rho'
    save_dir = '/work/mh1498/m301257/converted_data/processed/'
    grid_dict = {'nside': 256, 'nest': True, 'minmax_lat': 20}
    target_lat = xr.DataArray(np.arange(-20, 21, 2), dims='lat')
    target_lon = xr.DataArray(np.arange(0, 360, 2), dims='lon')
    
    # ✅ 修正：使用C5下的实验配置
    experiments = [
        {'name': 'cntl', 'key': 'AMIP_CNTL'},
        {'name': 'p4k', 'key': 'AMIP_P4K'},
        {'name': '4co2', 'key': 'AMIP_4CO2'}
    ]
    
    print("🎯 开始批量处理...")
    
    for exp in experiments:
        process_var_data(
            var_name, 
            exp['name'], 
            exp['key'], 
            save_dir, 
            grid_dict, 
            target_lat, 
            target_lon, 
            has_level=True, 
            level_slice=(30, None),
            max_levels=None,  # 处理所有层
            cat=cat
        )
    
    print("🎉 所有实验处理完成！")

print("✅ 主执行函数已加载")

In [ ]:
# 检查系统资源
import psutil

def check_system_resources():
    """检查并打印系统资源使用情况"""
    # 内存
    mem = psutil.virtual_memory()
    print("="*70)
    print("💻 系统资源状态")
    print("="*70)
    print(f"内存总量: {mem.total / (1024**3):.2f} GB")
    print(f"内存可用: {mem.available / (1024**3):.2f} GB")
    print(f"内存使用率: {mem.percent}%")
    
    # CPU
    cpu_percent = psutil.cpu_percent(interval=1)
    print(f"CPU使用率: {cpu_percent}%")
    print(f"CPU核心数: {psutil.cpu_count()}")
    
    # 磁盘
    disk = psutil.disk_usage('/work')
    print(f"磁盘总量: {disk.total / (1024**3):.2f} GB")
    print(f"磁盘可用: {disk.free / (1024**3):.2f} GB")
    print(f"磁盘使用率: {disk.percent}%")
    print("="*70)
    
    # 警告检查
    if mem.percent > 85:
        print("⚠️ 警告: 内存使用率过高！建议减小批次大小或重启内核")
    if disk.percent > 90:
        print("⚠️ 警告: 磁盘空间不足！")
    
    return mem.available / (1024**3)  # 返回可用内存(GB)

available_memory = check_system_resources()


In [ ]:
# 🚀 完整处理（测试成功后取消注释运行）

if __name__ ==  "__main__":
    from dask_jobqueue import SLURMCluster
    from dask.distributed import Client

    cluster = SLURMCluster(
        name="rho",
        queue="shared",
        memory="200GB",  # 增加内存限制
        cores=64,
        interface="ib0",
        account="mh1498",
        walltime="08:00:00",
        local_directory="/work/mh1498/m301257/dask-scratch-space",
        log_directory="/work/mh1498/m301257/dask-scratch-space/logs",
        scheduler_options={"dashboard_address": ":8788"},
    )
    client = Client(cluster)
    cluster.scale(1)
    print("Dashboard:", client.dashboard_link)
    main()

